# CCC Protein Sequence Extraction and GO Annotation Pipeline

This script extracts protein sequences containing CCC (conserved inter- and intra-species) from FASTA files, retrieves metadata from MongoDB, and enriches headers with species family and molecular function annotations from GO databases.

**Categories processed:**
- NCCC proteins (inter-species)
- NCCC proteins (intra-species)
- PCCC proteins (inter-species)
- PCCC proteins (intra-species)

## 1. Requirements and Database Configuration

In [ ]:
import pymongo
from pymongo import MongoClient
import pandas as pd
import statistics
import numpy as np
import os
from pathlib import Path
from collections import defaultdict

# MongoDB Connection
mongo_client = MongoClient("localhost", 27017)
print("MongoDB connection established")

## 2. Initialize Databases and Collections

In [ ]:
# Proteome databases
proteome_dbs = {
    'GANM': mongo_client['GANM'],
    'LPM': mongo_client['LPM'],
    'LPNM': mongo_client['LPNM']
}

# CCC cluster databases
ccc_databases = {
    'GANM': mongo_client['GANM_w25_ncc'],
    'LPM': mongo_client['LPM_w15_ncc'],
    'LPNM': mongo_client['LPNM_w20_ncc']
}

# Conserved cluster databases
conserved_db = mongo_client['Conserved_cluster_inter']
conserved_db_intra = mongo_client['Conserved_cluster_intra']

print("Database connections initialized")
print(f"Proteome databases: {list(proteome_dbs.keys())}")
print(f"CCC databases: {list(ccc_databases.keys())}")

## 3. Load GO Annotations (Molecular Function)

In [ ]:
def load_go_annotations(excel_path):
    """
    Load GO molecular function annotations from an Excel file.
    
    Args:
        excel_path (str): Path to the GO annotation Excel file
        
    Returns:
        dict: Dictionary mapping protein IDs to sets of molecular functions
    """
    try:
        df = pd.read_excel(excel_path, engine='openpyxl')
        
        # Fill missing Protein IDs with previous value
        df['Protein ID'].fillna(method='ffill', inplace=True)
        
        # Handle missing molecular functions
        df['Molecular Function'] = df['Molecular Function'].astype(str).fillna('Unknown')
        
        # Create protein ID -> functions mapping
        protein_to_functions = df.groupby('Protein ID')['Molecular Function'].apply(
            lambda x: set(f.strip() for f in x if f.strip() and f != 'Unknown')
        ).to_dict()
        
        print(f"Loaded GO annotations for {len(protein_to_functions)} proteins from {excel_path}")
        return protein_to_functions
    except FileNotFoundError:
        print(f"Warning: GO annotation file not found: {excel_path}")
        return {}

# Initialize empty annotation dictionary (to be filled with actual paths)
go_annotations = {}
print("GO annotation loader initialized. Update paths as needed.")

## 4. Core Functions for Protein Extraction

In [ ]:
def get_species_list(proteome_db):
    """
    Get list of species (collections) from a proteome database.
    
    Args:
        proteome_db: MongoDB database object
        
    Returns:
        list: List of collection names (species)
    """
    return proteome_db.list_collection_names()


def find_protein_by_id(proteome_dbs, protein_id):
    """
    Search for a protein ID across all proteome databases and return the document.
    
    Args:
        proteome_dbs (dict): Dictionary of proteome databases
        protein_id (str): Protein ID to search for
        
    Returns:
        tuple: (document, db_name, species_name) or (None, None, None) if not found
    """
    for db_name, db in proteome_dbs.items():
        species_list = get_species_list(db)
        for species in species_list:
            collection = db[species]
            doc = collection.find_one({'id': protein_id})
            if doc:
                return doc, db_name, species
    return None, None, None


def extract_ccc_proteins(ccc_cluster_collection, proteome_dbs, category_type):
    """
    Extract CCC protein IDs and metadata from cluster collection.
    
    Args:
        ccc_cluster_collection: MongoDB collection with cluster data
        proteome_dbs (dict): Dictionary of proteome databases
        category_type (str): Type of CCC ('NCCC' or 'PCCC')
        
    Returns:
        dict: Dictionary with protein metadata
    """
    protein_data = {}
    protein_lengths = []
    
    for cluster_doc in ccc_cluster_collection.find():
        cluster_id = cluster_doc.get('cluster')
        
        # Search for proteins matching this cluster
        for db_name, db in proteome_dbs.items():
            species_list = get_species_list(db)
            for species in species_list:
                collection = db[species]
                proteins = collection.find({'cluster': cluster_id})
                
                for protein in proteins:
                    protein_id = protein.get('id')
                    seq_len = protein.get('seq_len', 0)
                    
                    protein_data[protein_id] = {
                        'cluster': cluster_id,
                        'db_name': db_name,
                        'species': species,
                        'seq_len': seq_len
                    }
                    protein_lengths.append(seq_len)
    
    # Calculate statistics
    stats = {}
    if protein_lengths:
        stats['count'] = len(protein_data)
        stats['mean'] = statistics.mean(protein_lengths)
        stats['median'] = statistics.median(protein_lengths)
        stats['stdev'] = statistics.stdev(protein_lengths) if len(protein_lengths) > 1 else 0
        stats['q1'] = np.percentile(protein_lengths, 25)
        stats['q3'] = np.percentile(protein_lengths, 75)
        stats['iqr'] = stats['q3'] - stats['q1']
    
    return protein_data, stats


def build_fasta_header(protein_doc, db_name, species, protein_id, go_functions=None):
    """
    Build a comprehensive FASTA header with species family and molecular function.
    
    Header format:
    >{db_name}_{species}_{protein_id} | MF:{functions} | Description
    
    Args:
        protein_doc (dict): Protein document from MongoDB
        db_name (str): Database name (GANM, LPM, LPNM)
        species (str): Species name
        protein_id (str): Protein ID
        go_functions (set): Set of molecular functions from GO annotations
        
    Returns:
        str: Formatted FASTA header
    """
    header_parts = [f"{db_name}_{species}_{protein_id}"]
    
    # Add molecular function information
    if go_functions and len(go_functions) > 0:
        mf_str = ";".join(sorted(go_functions))
        header_parts.append(f"MF:{mf_str}")
    else:
        header_parts.append("MF:Unknown")
    
    # Add description if available
    description = protein_doc.get('description', '')
    if description:
        header_parts.append(f"Desc:{description}")
    
    # Add protein name if available
    prot_name = protein_doc.get('name', '')
    if prot_name:
        header_parts.append(f"Name:{prot_name}")
    
    return " | ".join(header_parts)


def write_fasta_file(output_path, protein_ids, protein_data_dict, proteome_dbs, go_annotations_dict):
    """
    Write proteins to FASTA file with enriched headers.
    
    Args:
        output_path (str): Path to output FASTA file
        protein_ids (list): List of protein IDs to write
        protein_data_dict (dict): Dictionary with protein metadata
        proteome_dbs (dict): Dictionary of proteome databases
        go_annotations_dict (dict): Dictionary of GO annotations
        
    Returns:
        int: Number of proteins successfully written
    """
    written_count = 0
    not_found_count = 0
    
    with open(output_path, 'w') as f:
        for protein_id in protein_ids:
            if protein_id not in protein_data_dict:
                not_found_count += 1
                continue
            
            metadata = protein_data_dict[protein_id]
            db_name = metadata['db_name']
            species = metadata['species']
            
            # Retrieve full protein document
            doc, _, _ = find_protein_by_id(proteome_dbs, protein_id)
            if not doc:
                not_found_count += 1
                continue
            
            sequence = doc.get('seq')
            if not sequence:
                continue
            
            # Get GO annotations
            go_functions = go_annotations_dict.get(protein_id, set())
            
            # Build header and write
            header = build_fasta_header(doc, db_name, species, protein_id, go_functions)
            f.write(f">{header}\n")
            f.write(f"{sequence}\n")
            written_count += 1
    
    print(f"Wrote {written_count} proteins to {output_path}")
    if not_found_count > 0:
        print(f"Warning: {not_found_count} proteins not found")
    
    return written_count


def print_stats(category_name, stats):
    """
    Print statistics for a protein category.
    
    Args:
        category_name (str): Name of the category
        stats (dict): Statistics dictionary
    """
    print(f"\n{'='*60}")
    print(f"Statistics for {category_name}")
    print(f"{'='*60}")
    print(f"Total proteins: {stats.get('count', 0)}")
    print(f"Mean length: {stats.get('mean', 0):.2f}")
    print(f"Median length: {stats.get('median', 0):.2f}")
    print(f"Std Dev: {stats.get('stdev', 0):.2f}")
    print(f"Q1: {stats.get('q1', 0):.2f}")
    print(f"Q3: {stats.get('q3', 0):.2f}")
    print(f"IQR: {stats.get('iqr', 0):.2f}")

## 5. Process NCCC Inter-Species Proteins

In [ ]:
# Load GO annotations for NCCC inter-species
# Update the path to your actual GO annotation file
nccc_inter_go_path = "GO_summary_NCCC_inter_proteins.xlsx"
nccc_inter_go = load_go_annotations(nccc_inter_go_path)

# Extract proteins
nccc_inter_collection = conserved_db['inter_specie_ncc']
nccc_inter_proteins, nccc_inter_stats = extract_ccc_proteins(
    nccc_inter_collection, 
    proteome_dbs,
    'NCCC'
)

print_stats("NCCC Inter-Species", nccc_inter_stats)

# Write FASTA file
nccc_inter_protein_ids = list(nccc_inter_proteins.keys())
nccc_inter_written = write_fasta_file(
    'NCCC_inter_proteins_annotated.fasta',
    nccc_inter_protein_ids,
    nccc_inter_proteins,
    proteome_dbs,
    nccc_inter_go
)

## 6. Process NCCC Intra-Species Proteins

In [ ]:
# Load GO annotations for NCCC intra-species
nccc_intra_go_path = "GO_summary_NCCC_intra_proteins.xlsx"
nccc_intra_go = load_go_annotations(nccc_intra_go_path)

# Extract proteins
nccc_intra_collection = conserved_db_intra['intra_specie_ncc']
nccc_intra_proteins, nccc_intra_stats = extract_ccc_proteins(
    nccc_intra_collection,
    proteome_dbs,
    'NCCC'
)

print_stats("NCCC Intra-Species", nccc_intra_stats)

# Write FASTA file
nccc_intra_protein_ids = list(nccc_intra_proteins.keys())
nccc_intra_written = write_fasta_file(
    'NCCC_intra_proteins_annotated.fasta',
    nccc_intra_protein_ids,
    nccc_intra_proteins,
    proteome_dbs,
    nccc_intra_go
)

## 7. Process PCCC Inter-Species Proteins

In [ ]:
# Load GO annotations for PCCC inter-species
pccc_inter_go_path = "GO_summary_PCCC_inter_proteins.xlsx"
pccc_inter_go = load_go_annotations(pccc_inter_go_path)

# Extract proteins from PCCC inter-species cluster collection
pccc_inter_collection = conserved_db['inter_specie_pccc']
pccc_inter_proteins, pccc_inter_stats = extract_ccc_proteins(
    pccc_inter_collection,
    proteome_dbs,
    'PCCC'
)

print_stats("PCCC Inter-Species", pccc_inter_stats)

# Write FASTA file
pccc_inter_protein_ids = list(pccc_inter_proteins.keys())
pccc_inter_written = write_fasta_file(
    'PCCC_inter_proteins_annotated.fasta',
    pccc_inter_protein_ids,
    pccc_inter_proteins,
    proteome_dbs,
    pccc_inter_go
)

## 8. Process PCCC Intra-Species Proteins

In [ ]:
# Load GO annotations for PCCC intra-species
pccc_intra_go_path = "GO_summary_PCCC_intra_proteins.xlsx"
pccc_intra_go = load_go_annotations(pccc_intra_go_path)

# Extract proteins from PCCC intra-species cluster collection
pccc_intra_collection = conserved_db_intra['intra_specie_pccc']
pccc_intra_proteins, pccc_intra_stats = extract_ccc_proteins(
    pccc_intra_collection,
    proteome_dbs,
    'PCCC'
)

print_stats("PCCC Intra-Species", pccc_intra_stats)

# Write FASTA file
pccc_intra_protein_ids = list(pccc_intra_proteins.keys())
pccc_intra_written = write_fasta_file(
    'PCCC_intra_proteins_annotated.fasta',
    pccc_intra_protein_ids,
    pccc_intra_proteins,
    proteome_dbs,
    pccc_intra_go
)

## 9. Output Files Generated

The following FASTA files have been created with annotated headers:

1. **NCCC_inter_proteins_annotated.fasta** 
2. **NCCC_intra_proteins_annotated.fasta** 
3. **PCCC_inter_proteins_annotated.fasta** 
4. **PCCC_intra_proteins_annotated.fasta** 
5. **CCC_extraction_summary.csv** - Summary statistics for all categories

### Header Format
```
>GANM_Genus_species_ProteinID | MF:Molecular_Function1;Molecular_Function2 | Desc:Description | Name:ProteinName
```

Each header contains:
- **Database and species family**: `GANM/LPM/LPNM_Genus_species`
- **Protein ID**: Unique identifier
- **Molecular Functions (MF)**: GO annotations separated by semicolons
- **Description**: Protein description from proteome database
- **Name**: Official protein name